In [ ]:
!pip install -U tensorly
!pip install numpy
!pip install sparse
!pip install torch

     |████████████████████████████████| 198 kB 14.3 MB/s 
     |████████████████████████████████| 154 kB 73.6 MB/s 
     |████████████████████████████████| 77 kB 4.4 MB/s 


In [ ]:

import tensorly as tl
from os.path import exists
import sklearn
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib as plt
from tensorly.decomposition import RandomizedCP
# from tensorly.decomposition._nn_cp import initialize_nn_cp
from tensorly.cp_tensor import CPTensor
from tensorly import random
import time
from copy import deepcopy
import pickle
import sparse
from psutil import virtual_memory
from numpy.random import default_rng
from tensorly.contrib.sparse.decomposition import parafac as sparse_parafac
from numpy.linalg import norm
import math
from numpy import r_
import tensorly.tenalg as tl_alg
import numpy.matlib as matlib
import warnings
import math

from multiprocessing import freeze_support
from multiprocessing import Pool
from functools import partial
import threading


warnings.filterwarnings("ignore", category=DeprecationWarning)

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

np.random.seed(1234)

Your runtime has 27.3 gigabytes of available RAM

You are using a high-RAM runtime!


In [ ]:
def cost(X0, norm_x, A):
    X_bar = A[0] @ (tl_alg.khatri_rao(A, skip_matrix=0).T)
    return norm(X0 - X_bar) / norm_x


def MSE(X, Xt):
    X1 = X @ np.diag(np.divide(1, np.sqrt(np.sum(np.square(X), axis=0))))

    # Xt = Xt*diag(1./(sqrt(sum(Xt.^2))+eps));

    X2 = Xt @ np.diag(
        np.divide(1, np.sqrt(np.sum(np.square(Xt), axis=0) + np.finfo(float).eps))
    )

    M = np.transpose(X1) @ X2

    MSE_col = np.max(M)
    MSE = np.abs(np.mean(2 - 2 * MSE_col))

    return MSE

def initDecomposition(Sizes, F):
    # initialize the latent factors
    Hinit = []
    for d in range(3):
        Hinit.append(np.random.random((Sizes[d], F)))
    return Hinit

def createTensor(size,F):
    X = np.zeros((size, size, size))

    # generate true latent factors
    A = []
    for i in range(3):
        A.append(np.random.random((size, F)))

    A_gt = A

    # form the tensor
    for k in range(size):
        X[:, :, k] = A[0] @ np.diag(A[2][k, :]) @ np.transpose(A[1])
    return X

def sample_fibers(n_fibers, dim_vec, d):
    dim_len = len(dim_vec)
    tensor_idx = np.zeros((n_fibers, dim_len))

    # randomly select fibers for dimensions not d
    for i in r_[0 : d, d + 1 : dim_len]:
        tensor_idx[:, i] = np.random.randint(
            0, high=dim_vec[i] - 1, size=(n_fibers)
        )

    factor_idx = tensor_idx[:, r_[0 : d, d + 1 : dim_len]]

    tensor_idx = tl.tenalg.kronecker([tensor_idx, np.ones((dim_vec[d], 1))])

    tensor_idx[:, d] = (
        matlib.repmat(np.arange(0, dim_vec[d]).transpose(), n_fibers, 1)
    ).reshape(tensor_idx[:, d].shape)

    return (tensor_idx, factor_idx)


def sampled_kr(A_unsel, factor_idx):
    l = [x for x in range(len(A_unsel) - 1, -1, -1)]
    H = [A_unsel[l[0]][int(x), :] for x in factor_idx[:, l[0]]]
    for i in l[1:]:
        H = np.multiply(H, np.array([A_unsel[i][int(x), :] for x in factor_idx[:, i]]))
        H = np.array(H)
    return H


def lookup(x, idx):
    ret = list(x)
    for i in idx:
        ret = ret[int(i)]
    return np.array(ret)

def proxr(Hb, d):
    return Hb

def error(X0, norm_x, A, B, C):
    X_bar = A @ (tl_alg.khatri_rao([A, B, C], skip_matrix=0).T)
    return tl.norm(X0 - X_bar)/norm_x

In [ ]:
# /My Drive/Senior Year/Graph Mining/Mount for Colab/rawdatafortensor-gms22data-QueryResult.csv


FileNotFoundError: ignored

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/rawdatafortensor-gms22data-QueryResult_mindegree69.csv', sep=',')
df['b_id'] = pd.factorize(df['beer_id'])[0]
df['u_id'] = pd.factorize(df['user'])[0] + df['b_id'].nunique()

In [ ]:

N= df['u_id'].nunique() +  df['b_id'].nunique()
print(N)
tensor = np.zeros((N,N,5), dtype=np.float16)
mask =  np.zeros((N,N,5), dtype=bool)
train, test = train_test_split(df, test_size=0.1)
train, validation = train_test_split(train, test_size=0.125)

# tensor = np.load(f'/content/drive/MyDrive/tensor_data.npy')

8051


In [ ]:
one_v_five = np.array([1]*5)
for i in range(len(train)):
  row = train.iloc[i]
  b,u = row['b_id'], row['u_id']
  review_vector = np.array([row['review_overall'],row['review_aroma'],row['review_appearance'],row['review_palate'], row['review_taste']])
  tensor[u][b] = review_vector/5
  tensor[b][u] = review_vector/5
  mask[u][b] = one_v_five
  mask[b][u] = one_v_five
  if i % 100000 == 0: print(f"{i}/{len(train)}", end='\r')



In [ ]:
def AdaCPD(X, b0, n_mb, max_it, min_itr, rel_error_stop, A_init, mask = None, eta=1):
    A = A_init

    # setup parameters
    dim = len(X.shape)
    dim_vec = X.shape

    F = A[0].shape[1]

    PP = tl.kruskal_to_tensor((np.ones(F), A))

    X_norm = np.linalg.norm(X)

    mask_norm = np.linalg.norm(mask)**2

    err_e = np.linalg.norm(X - mask*PP) ** 2 / mask_norm
    NRE_A = [err_e]
    Rel_Err = []

    mu = 0

    Gt = [b0 for _ in range(dim)]

    mttrk = 0
    for it in range(1, max_it+1):
        # randomly permute the dimensions
        block_vec = np.random.permutation(dim)
        d_update = block_vec[0]

        # sampling fibers and forming the X_[d] = H_[d] A_[d]^t least squares
        [tensor_idx, factor_idx] = sample_fibers(n_mb, dim_vec, d_update)
        tensor_idx = tensor_idx.astype(int)

        cols = [tensor_idx[:, x] for x in range(len(tensor_idx[0]))]
        X_sample = X[tuple(cols)]
        X_sample = X_sample.reshape((int(X_sample.size / dim_vec[d_update]), dim_vec[d_update]))
        if mask is not None:
          mask_sample = mask[tuple(cols)]
          mask_sample = mask_sample.reshape((int(mask_sample.size / dim_vec[d_update]), dim_vec[d_update]))

        # perform a sampled khatrirao product
        A_unsel = []
        for i in range(d_update):
            A_unsel.append(A[i])
        for i in range(d_update + 1, dim):
            A_unsel.append(A[i])
        H = np.array(sampled_kr(A_unsel, factor_idx))

        # if mask is not None:
        #   # H = H.reshape(int(len(H)/F), F)
        #   print(H.shape)
        #   print(mask_sample.shape)
        #   print(X_sample.shape)
        #   g = 2 * H @ (mask_sample * (H@ A[d_update].T)) - 2 * H.T @ (mask_sample * X_sample)
        # else:
          # compute the gradient
        g = (1 / n_mb) * (
            (mask_sample.T * (A[d_update] @ H.T)) @ H - X_sample.T @ H
        )

        # compute the accumlated gradient
        Gt[d_update] = np.abs(np.square(g)) + Gt[d_update]

        eta_adapted = np.divide(eta, np.sqrt(Gt[d_update]))
        d = d_update
        A[d_update] = A[d_update] - np.multiply(eta_adapted, g)

        A[d_update] = proxr(A[d_update], d)

        PP = tl.kruskal_to_tensor((np.ones(F), A))
        error = np.linalg.norm(X - mask*PP) ** 2 / mask_norm
        NRE_A.append(error)
        rel_error = (NRE_A[-2] - NRE_A[-1])/NRE_A[-1]
        Rel_Err.append(rel_error)

        print(f"{it:4d}: rel_error: {rel_error:0.4f}, error: {error:0.4f}, rank {F:4d}")
        time.sleep(0)
        if (it >= 10 and it >= min_itr) and sum(Rel_Err[-5:])/5 < rel_error_stop: break
    return (NRE_A,A)

In [ ]:
np.save('/content/tensor_data', tensor)
np.save('/content/mask_data', tensor)
with open( "/content/train_test_validate_data", "wb") as f:
  pickle.dump((train, test, validation), f)

In [ ]:
def sampledALS(X, rank, n_samples, max_itr):
  tl.set_backend('pytorch')
  t = tl.tensor(X)
  t = t.to('cuda')

  decomp = RandomizedCP(rank, n_samples, max_itr, verbose=4)
  factors = decomp.fit_transform(t)
  return  tl.to_numpy(tl.cp_to_tensor(factors))
  int(math.sqrt(rank) * 500)

In [ ]:
 #convert to tensor
# tl.set_backend('pytorch', local_threadsafe=False)
# tensor = tl.tensor(tensor)#.to('cuda')

In [ ]:
ranks = np.linspace(1,25, 24)
decomps_per_rank = 2

In [ ]:
def job(data):
  print(f"Starting: {data}")
  rank, i = data

  tic = time.time()
  err, approx = AdaCPD(tensor, .1, 5000, 300, 100, 10**-3, initDecomposition([N,N,5], int(rank)), mask=mask, eta=1)
  with open(f"/content/approximations/{int(rank)}_{i}_non-negative-mask", "wb") as f:
    pickle.dump(approx, f)

  PP = tl.kruskal_to_tensor((np.ones(rank), approx))

  time_mu = time.time()-tic
  validation_error = testError(PP, validation)
  test_error = testError(PP, test)
  print(f"rank: {rank} i: {i}  error: {err[0]} -> {err[-1]} time:{time_mu} validation error:{validation_error}")

  return validation_error, test_error

In [ ]:
def testError(approx, test):
  errors_sum = 0
  for i in range(len(test)):
    row = test.iloc[i]
    b,u = row['b_id'], row['u_id']
    review_vector = np.array([row['review_overall'],row['review_aroma'],row['review_appearance'],row['review_palate'], row['review_taste']])
    estimate = .5 * (5 * approx[u][b] + 5 * approx[b][u])
    point_error = np.log(norm(review_vector - estimate))**2
    errors_sum += point_error
  return 1/len(test) * errors_sum

In [ ]:
from multiprocessing import Pool

jobs = [(int(rank), i) for i in range(decomps_per_rank) for rank in ranks]

p = Pool(5, maxtasksperchild=1)

results = p.map(job, jobs)

with open(f"/content/outOfSamples{time.time()}", "wb") as f:
  pickle.dump((jobs, results))

Starting: (1, 0)
Starting: (4, 0)
Starting: (7, 0)
Starting: (13, 0)
Starting: (10, 0)
Starting: (16, 0)
Starting: (19, 0)
Starting: (22, 0)
Starting: (1, 1)
Starting: (4, 1)
Starting: (7, 1)
Starting: (10, 1)
Starting: (13, 1)
Starting: (16, 1)
Starting: (19, 1)
Starting: (22, 1)
   1: rel_error: 0.1269, error: 0.9815, rank   13
   1: rel_error: 0.4973, error: 1.4214, rank   16
   1: rel_error: 0.2234, error: 4.3505, rank   22
   2: rel_error: 0.3119, error: 0.7482, rank   13
   2: rel_error: 0.3295, error: 1.0691, rank   16
   3: rel_error: 0.0923, error: 0.6850, rank   13
   4: rel_error: 0.2339, error: 0.5551, rank   13
   3: rel_error: 0.2524, error: 0.8536, rank   16
   4: rel_error: 0.0595, error: 0.8057, rank   16
   5: rel_error: 0.0571, error: 0.7622, rank   16
   6: rel_error: 0.0614, error: 0.7181, rank   16
   5: rel_error: 0.1842, error: 0.4688, rank   13
   7: rel_error: 0.0829, error: 0.6631, rank   16
   6: rel_error: 0.0881, error: 0.4309, rank   13
   8: rel_error: 0

In [ ]:
p.terminate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/MyDrive/results", "wb") as f:
  pickle.dump((jobs, results, train, test, validation, tensor, mask))